# Validation Test Execution Notebook
### Air Quality Review (AQR) System â€” GAMP 5 Category 5 Validation Protocol

This notebook contains individual cells to execute and verify every test case in the combined validation protocol (Appendix 2 and 3).
All test data is prepared by copying and programmatically editing real files from the project's `data/` directory to ensure high-fidelity testing without mock templates.

## Setup & Helpers

In [60]:
import sys
import os
import shutil
import pandas as pd
import numpy as np
from datetime import datetime

# Ensure project root is in system path
project_dir = r'D:\ex_work\AirQualityReview_Project'
if project_dir not in sys.path:
    sys.path.append(project_dir)

import analysis_logic
import audit_trail

# Setup test data workspace directory
test_workspace = os.path.join(project_dir, 'data', 'validation_tests')
os.makedirs(test_workspace, exist_ok=True)
print("Validation test workspace configured at:", test_workspace)

Validation test workspace configured at: D:\ex_work\AirQualityReview_Project\data\validation_tests


# Part 1: Module & Unit Verification (Appendix 2)

### MT-01: Software Installation & Env Verification

In [61]:
print("--- Executing MT-01: Environment Initialization & Self-Healing ---")

# 1. Verify existence of build files
exe_info_path = os.path.join(project_dir, 'app_version_info.txt')
if os.path.exists(exe_info_path):
    with open(exe_info_path, 'r') as f:
        print("App Version Info Contents:\n", f.read().strip())

# 2. Test self-healing: Clear logs and reports contents to verify they are recreated/usable
temp_reports = os.path.join(project_dir, 'reports')
temp_logs = os.path.join(project_dir, 'logs')

for path in [temp_reports, temp_logs]:
    if os.path.exists(path):
        for f in os.listdir(path):
            f_path = os.path.join(path, f)
            try:
                if os.path.isfile(f_path): os.remove(f_path)
            except Exception as e: print(f"Skipping file {f}: {e}")

print("Cleared logs and reports folder contents to test self-healing.")

# Trigger recreations
os.makedirs(temp_reports, exist_ok=True)
os.makedirs(temp_logs, exist_ok=True)
audit_trail.log_event("TEST_SELF_HEAL", "Folders successfully self-healed")

print("Is reports folder present?:", os.path.exists(temp_reports))
print("Is logs folder present?:", os.path.exists(temp_logs))
print("Is audit_trail.log created?:", os.path.exists(os.path.join(temp_logs, 'audit_trail.log')))

# .venv\Scripts\Activate.ps1
# pyinstaller AQR_Dashboard_v1.1.0_Fix.spec

--- Executing MT-01: Environment Initialization & Self-Healing ---
App Version Info Contents:
 # IQ-TC-01: Binary Integrity & Versioning Verification
#
# For more details about fixed file info:
# http://msdn.microsoft.com/en-us/library/ms646997.aspx
VSVersionInfo(
  ffi=FixedFileInfo(
    filevers=(1, 1, 0, 0),
    prodvers=(1, 1, 0, 0),
    mask=0x3f,
    flags=0x0,
    OS=0x40004,
    fileType=0x1,
    subtype=0x0,
    date=(0, 0)
    ),
  kids=[
    StringFileInfo(
      [
      StringTable(
        u'040904B0',
        [StringStruct(u'CompanyName', u'AQR Program'),
        StringStruct(u'FileDescription', u'Air Quality Review System - GAMP 5 Compliant'),
        StringStruct(u'FileVersion', u'1.1.0'),
        StringStruct(u'InternalName', u'AirQualityReview'),
        StringStruct(u'LegalCopyright', u'Ã‚Â© 2026 AQR Program. All rights reserved.'),
        StringStruct(u'OriginalFilename', u'AirQualityReview_1.1.0.exe'),
        StringStruct(u'ProductName', u'Air Quality Review'),
 

### MT-02: Cryptographic Hashing (`get_file_hash`)

In [62]:
print("--- Executing MT-02: File Hash Traceability ---")

# Setup a temporary file
test_file = os.path.join(test_workspace, 'mt02_test.txt')
with open(test_file, 'w') as f:
    f.write("Valid calibration data context.")

# Calculate hash
hash_orig = analysis_logic.get_file_hash(test_file)
print("Original File Hash:", hash_orig)
assert len(hash_orig) == 64, "Hash must be 64 characters hexadecimal"

# Modify file
with open(test_file, 'w') as f:
    f.write("Tampered calibration data context.")
hash_mod = analysis_logic.get_file_hash(test_file)
print("Modified File Hash:", hash_mod)
assert hash_orig != hash_mod, "Hash must change upon file modification"

# Test invalid file path
err_hash = analysis_logic.get_file_hash(os.path.join(test_workspace, 'non_existent.txt'))
print("Invalid File Hash Response:", err_hash)
assert err_hash.startswith("ERROR:"), "Must return an error message string"


--- Executing MT-02: File Hash Traceability ---
Original File Hash: 3852bbdd4e7173492eb07202e62c8622a2435d51f456fe7512cf58b57154f25c
Modified File Hash: f462b990918f748df0654e7d80b6b358df8803b919362213784cffd73459902a
Invalid File Hash Response: ERROR: [Errno 2] No such file or directory: 'D:\\ex_work\\AirQualityReview_Project\\data\\validation_tests\\non_existent.txt'


### MT-03: Header Row Detection (`find_header`)

In [63]:
print("--- Executing MT-03: find_header ---")

# Header at index 0
lines_1 = ["<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_1 = analysis_logic.find_header(lines_1)
print("Header at beginning row index:", idx_1)
assert idx_1 == 0

# Header after multiple rows
lines_2 = ["BAS System Log File", "Generated on Friday", "<>Date,Time,Point_1", "05/30/2026,00:00,22.5"]
idx_2 = analysis_logic.find_header(lines_2)
print("Header after multiple rows index:", idx_2)
assert idx_2 == 2

# Missing header
lines_3 = ["Wrong header row name", "Time,Value"]
idx_3 = analysis_logic.find_header(lines_3)
print("Missing header index response:", idx_3)
assert idx_3 is None

print('---------------')
for i in lines_1:
    print(i)
print('---------------')
for i in lines_1:
    print(i)

--- Executing MT-03: find_header ---
Header at beginning row index: 0
Header after multiple rows index: 2
Missing header index response: None
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5
---------------
<>Date,Time,Point_1
05/30/2026,00:00,22.5


### MT-04: Dynamic Room Point Mapping (`find_point_mapping`)

In [64]:
print("--- Executing MT-04: find_point_mapping ---")

headers = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1P040 ROOM TEMP","","5 minutes"',
    '"Point_2:","1P040 ROOM .RMH","","5 minutes"',
    '"Point_3:","1P040 ROOM PRES","","5 minutes"',

]

headers_2 = [
    '"Key            Name:Suffix                                Trend Definitions Used"',
    '"Point_1:","1-P040 ROOM PRES","","5 minutes"',
]

print('-----------------')
test_01 = analysis_logic.find_point_mapping(headers, "1-P040", "TEMP")
print("Return:", test_01)

test_02 = analysis_logic.find_point_mapping(headers, "1-P040", ".RMH")
print("Return:", test_02)

test_03 = analysis_logic.find_point_mapping(headers, "1-P040", "HUM")
print("Return:", test_03)

print('-----------------')
test_04 = analysis_logic.find_point_mapping(headers_2, "1-P999", "PRES")
print("Return:", test_04)

print('-----------------')
test_05 = analysis_logic.find_point_mapping("TEMP", "1P040", "PRES")
print("Return:", test_05)

# print('-----------------')
# test_05 = analysis_logic.find_point_mapping()
# print("Return:", test_05)

--- Executing MT-04: find_point_mapping ---
-----------------
Return: Point_1
Return: Point_2
Return: Point_2
-----------------
Return: None
-----------------
Return: None


### MT-05: Continuous Index Sequence Delineation (`find_continuous_ranges`)

In [65]:
print("--- Executing MT-05: find_continuous_ranges ---")

# Sequential values
seq = [1, 2, 3, 5, 6, 10, 11, 13]
seq_2 = [1, 3, 5, 7, 9, 11]
seq_3 = []
ranges = analysis_logic.find_continuous_ranges(seq, min_length=2)
print("Continuous ranges:", ranges)
assert ranges == [(1, 3), (5, 6), (10, 11)]

ranges_2 = analysis_logic.find_continuous_ranges(seq_2, min_length=2)
print("Un-continuous ranges:", ranges_2)

# Empty list
ranges_3 = analysis_logic.find_continuous_ranges(seq_3, min_length=2)
print("Empty ranges:", ranges_3)

--- Executing MT-05: find_continuous_ranges ---
Continuous ranges: [(1, 3), (5, 6), (10, 11)]
Un-continuous ranges: []
Empty ranges: []


### MT-06: Date Bound Extraction (`get_file_date_range`)

In [66]:
print("--- Executing MT-06: get_file_date_range ---")

# Copy standard file and check
csv_01 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
csv_02 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P041_05-14-26_01-00.csv"
csv_03 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P042_05-14-26_01-00.csv"
csv_04 = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P043_05-14-26_01-00.csv"
# shutil.copyfile(src_csv, dest_csv)

start_d, end_d = analysis_logic.get_file_date_range(csv_01)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_02)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_03)
print(f"Parsed valid range: {start_d} to {end_d}")

start_d, end_d = analysis_logic.get_file_date_range(csv_04)
print(f"Parsed valid range: {start_d} to {end_d}")

--- Executing MT-06: get_file_date_range ---
Parsed valid range: 2026-05-29 to 2026-05-31
Parsed valid range: 2026-05-31 to 2026-05-31
Parsed valid range: 2001-01-01 to 2099-12-31
Parsed valid range: None to None


### MT-07: Standardized DataFrame Preparation (`prepare_df`)

In [67]:
print("--- Executing MT-07: prepare_df ---")

src_csv = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P040_05-14-26_01-00.csv"
limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path)

df = analysis_logic.prepare_df(src_csv, "1-P040", setpoint_df)
print("Reindexed DataFrame Columns:", df.columns.tolist())
print("Types in DataFrame Temperature:", df['Temperature'].dtype)
print("First 5 rows:\n", df.head(10))
assert 'Temperature' in df.columns
assert 'Humidity' in df.columns
assert 'Pressure' in df.columns


--- Executing MT-07: prepare_df ---
Reindexed DataFrame Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Types in DataFrame Temperature: float64
First 5 rows:
 0            DateTime  Temperature  Humidity  Pressure
0 2026-05-29 00:00:00         21.5      41.3      12.9
1 2026-05-30 00:05:00         21.5      41.5      12.9
2 2026-05-31 00:10:00         21.5      41.5      12.9


### MT-08: Pressure Corridor Mapping (`find_compare_path`)

In [68]:
print("--- Executing MT-08: find_compare_path ---")

limit_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"
setpoint_df = pd.read_excel(limit_path).dropna(subset=['Room_number'])

file_list = [
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv",
    r"D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P999_01-05-26_01-00.csv"
]

# Room 1-P040 comparison corridor is 1-P051
comp_room, comp_path = analysis_logic.find_compare_path(file_list, setpoint_df, "1-P040")
print(f"Room 1-P040 Comparison Room: {comp_room}")
print(f"Comparison File Path: {comp_path}")
# assert comp_room == "1-P051"
# assert comp_path is not None


--- Executing MT-08: find_compare_path ---
Room 1-P040 Comparison Room: 1-P051
Comparison File Path: D:\ex_work\AirQualityReview_Project\data\csv_main\C\1-P051_01-05-26_01-00.csv


### MT-09: Legacy Date Parsing & Offset (`parse_filename_for_datetime`)

In [69]:
print("--- Executing MT-09: parse_filename_for_datetime ---")

filename = "1-P040_05-30-26_00-00.csv"
parsed_date = analysis_logic.parse_filename_for_datetime(filename)
print("Filename: ", filename)
print("Parsed previous-day business date: ", parsed_date)
# 05-30-26 parsed is 2026-05-30. Previous day rule makes it 2026-05-29
assert parsed_date == datetime(2026, 5, 29).date()


--- Executing MT-09: parse_filename_for_datetime ---
Filename:  1-P040_05-30-26_00-00.csv
Parsed previous-day business date:  2026-05-29


### MT-10: Phase II Unified Data Generation (`prepare_df_phase2`)

In [70]:
print("--- Executing MT-10: prepare_df_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b11\2026-05-01"
limit_path_p2 = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit_B11.xlsx"
setpoint_df_p2 = pd.read_excel(limit_path_p2)

room_id, df_p2, sensors = analysis_logic.prepare_df_phase2(phase2_room_dir, "11-1-012", setpoint_df_p2)
print("Unified Room ID:", room_id)
print("Merged Columns:", df_p2.columns.tolist())
print("Discovered Sensors:", sensors)
display(df_p2.head())
assert 'Temperature' in df_p2.columns
assert 'Humidity' in df_p2.columns
assert 'Pressure' in df_p2.columns


--- Executing MT-10: prepare_df_phase2 ---


Unified Room ID: 11-1-012
Merged Columns: ['DateTime', 'Temperature', 'Humidity', 'Pressure']
Discovered Sensors: {'Temperature', 'Pressure', 'Humidity'}


,DateTime,Temperature,Humidity,Pressure
0,2026-04-30 08:00:00,22.47,49.04,15.46
1,2026-04-30 08:05:00,22.44,48.53,14.12
2,2026-04-30 08:10:00,22.44,48.53,16.90
3,2026-04-30 08:15:00,22.40,48.79,15.92
4,2026-04-30 08:20:00,22.40,48.79,15.94


### MT-11: Phase II Multiple Files Date Extraction (`get_file_date_range_phase2`)

In [71]:
print("--- Executing MT-11: get_file_date_range_phase2 ---")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b12"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-073")
print(f"Phase II date range: {s_date} to {e_date}")

s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-074")
print(f"Phase II date range: {s_date} to {e_date}")

phase2_room_dir = r"D:\ex_work\AirQualityReview_Project\data\csv_b12_blank"
s_date, e_date = analysis_logic.get_file_date_range_phase2(phase2_room_dir, "12-1-074")
print(f"Phase II date range: {s_date} to {e_date}")

--- Executing MT-11: get_file_date_range_phase2 ---
Phase II date range: 2026-05-29 to 2026-05-31
Phase II date range: 2026-05-29 to 2026-05-31
Phase II date range: None to None


# Part 2: Integration, System Transformation & Error Verification (Appendix 3)

### ERR-001: Missing Header Row Detection

In [72]:
print("--- Executing ERR-001: Missing header check ---")

err_dir = os.path.join(test_workspace, 'case_err001')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file missing <>Date
with open(err_file, 'w') as f:
    f.write('BAS Log file\nWrongHeaderRow,Time,Point_1\n05/30/2026,00:00,22.5\n')

try:
    analysis_logic.prepare_df(err_file, "1-P045")
    print("FAIL: Expected error was not raised")
except ValueError as e:
    print("PASS: Expected error message caught:", e)
    assert "ERR-001" in str(e)


--- Executing ERR-001: Missing header check ---
PASS: Expected error message caught: ERR-001: Critical Error - Header '<>Date' not found in file: D:\ex_work\AirQualityReview_Project\data\validation_tests\case_err001\1-P040_05-30-26_00-00.csv


### ERR-002: Missing Limit File Detection

In [73]:
print("--- Executing ERR-002: Missing limit file ---")

folder_path = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C"
missing_limit_path = r"D:\ex_work\AirQualityReview_Project\data\MissingLimit.xlsx"

out, logs, plot = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=missing_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Analysis return output path:", out)
print("Log output:\n", logs)
assert out is None
assert "ERR-002" in logs


--- Executing ERR-002: Missing limit file ---
Analysis return output path: None
Log output:
 ERROR: ERR-002: Limit File Not Found
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 792, in analyze_files
    setpoint_df = pd.read_excel(setpoint_path)
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 481, in read_excel
    io = ExcelFile(
        io,
    ...<2 lines>...
        engine_kwargs=engine_kwargs,
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1604, in __init__
    ext = inspect_excel_format(
        content_or_path=path_or_buffer, storage_options=storage_options
    )
  File "d:\ex_work\AirQualityReview_Project\.venv\Lib\site-packages\pandas\io\excel\_base.py", line 1452, in inspect_excel_format
    with get_handle(
         ~~~~~~~~~~^
        content_or_path, "rb", storage_options=storage_options, is_text=False
     

### ERR-003: Invalid Configuration DataType (Non-Numeric)

In [74]:
print("--- Executing ERR-003: Non-numeric limit checks ---")

err_limit_dir = os.path.join(test_workspace, 'case_err003')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err003.xlsx')

# Load standard limit and inject non-numeric
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit['Temperature_Limit'] = df_limit['Temperature_Limit'].astype(object)
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Temperature_Limit'] = "NonNumericLimit"
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
# assert "ERR-003" in logs, "Logs should capture the non-numeric limit GxP warning"


--- Executing ERR-003: Non-numeric limit checks ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-004: Audit Trail Tampering Detection

In [75]:
print("--- Executing ERR-004: Audit log tampering ---")

log_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log"
backup_file = r"D:\ex_work\AirQualityReview_Project\logs\audit_trail.log.tmp_bak"

# 1. Backup log
if os.path.exists(log_file):
    shutil.copyfile(log_file, backup_file)
    
    # 2. Tamper log
    with open(log_file, 'a', encoding='utf-8') as f:
        f.write('{"timestamp": "2026-06-04 10:00:00", "user": "attacker", "action": "TAMPER", "prev_hash": "wrong", "entry_hash": "invalid"}\n')

    try:
        valid, msg = audit_trail.verify_audit_trail()
        print(f"Verification status: {valid} | Msg: {msg}")
        assert not valid, "Validation must detect tampering"
        print("PASS: Tampering successfully detected.")
    finally:
        # 3. Restore backup
        shutil.copyfile(backup_file, log_file)
        os.remove(backup_file)
        print("Restored audit log backup.")


--- Executing ERR-004: Audit log tampering ---
Verification status: False | Msg: Broken chain at line 5: Hash mismatch.
PASS: Tampering successfully detected.
Restored audit log backup.


### ERR-005: Missing Parameter Raw Data or Columns

In [76]:
print("--- Executing ERR-005: Upfront missing raw files & columns validation ---")

# Test Phase I Missing raw file
out1, logs1, plot1 = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P032"],  # Exists in limit but no CSV in folder
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Phase I Missing file output logs:\n", logs1)
assert out1 is None
assert "ERR-005" in logs1

# Test Phase II Missing sensor files
dummy_scan_dir = os.path.join(test_workspace, 'dummy_p2')
os.makedirs(os.path.join(dummy_scan_dir, '1-P040'), exist_ok=True)
# Only humidity file exists, temp (RMT) is missing
with open(os.path.join(dummy_scan_dir, '1-P040', '1-P040_RMH_2026-05-30_Mock.csv'), 'w') as f:
    f.write("DateTime;Data Source;Value\n2026-05-30 00:00:00;Room Hum;45.2\n")

out2, logs2, plot2 = analysis_logic.analyze_files_phase2(
    folder_path=dummy_scan_dir,
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(dummy_scan_dir)
print("Phase II Missing sensor file logs:\n", logs2)
assert out2 is None
assert "ERR-005" in logs2


--- Executing ERR-005: Upfront missing raw files & columns validation ---
Phase I Missing file output logs:
 ERROR: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 861, in analyze_files
    raise ValueError(f"ERR-005: Raw data file not found in {folder_path} for Room {room_id}")
ValueError: ERR-005: Raw data file not found in D:\ex_work\AirQualityReview_Project\data\csv_main\C for Room 1-P032

Phase II Missing sensor file logs:
 FILE ERROR [1-P040]: ERR-005: No Temperature file (_RMT_) found in D:\ex_work\AirQualityReview_Project\data\validation_tests\dummy_p2 for 1-P040
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 1335, in analyze_files_phase2
    _, df, sensors = prepare_df_phase2(raw_data_path, room_id=room_id, setpoint_df=setpoint_df)
                     ~~~~~~~~~~~

### ERR-006: Inverted Limits Configuration (High < Low)

In [77]:
print("--- Executing ERR-006: Logical constraint check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err006')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err006.xlsx')

# Create limit file with Low Limit > High Limit
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_High_Limit'] = 20
df_limit.loc[df_limit['Room_number'] == '1-P040', 'Humidity_Low_Limit'] = 50
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is not None, "Output path should not be None since it completes Excel generation with logged error"
# assert "ERR-006" in logs, "Logs should capture GxP configuration limit inversion warning"


--- Executing ERR-006: Logical constraint check ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-007: Excel Report Generation Failure

In [78]:
print("--- Executing ERR-007: Simulate Report Write Lock ---")

# We mock standard ExcelWriter to raise PermissionError and verify ERR-007
import unittest.mock as mock

with mock.patch('pandas.ExcelWriter', side_effect=Exception("Write Lock Permission Denied")):
    out, logs, plot = analysis_logic.analyze_files(
        folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
        setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
        selected_rooms=["1-P040"],
        start_date="2026-05-13",
        end_date="2026-05-13"
)
print("Log output:\n", logs)
# assert out is None
# assert "ERR-007" in logs


--- Executing ERR-007: Simulate Report Write Lock ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-008: Identical Duplicate Timestamps

In [79]:
print("--- Executing ERR-008: Duplicate Timestamp Deduplication ---")

err_dir = os.path.join(test_workspace, 'case_err008')
os.makedirs(err_dir, exist_ok=True)
err_file = os.path.join(err_dir, '1-P040_05-30-26_00-00.csv')

# Create file with identical duplicate timestamps
csv_dup = (
    '"Key            Name:Suffix                                Trend Definitions Used"\n'
    '"Point_1:","1-P040 ROOM TEMP","","5 minutes"\n'
    '"Point_2:","1-P040 ROOM HUM","","5 minutes"\n'
    '"Point_3:","1-P040 ROOM PRES","","5 minutes"\n'
    '"<>Date","Time","Point_1","Point_2","Point_3"\n'
    '"05/30/2026","00:00","22.5","45.0","40.0"\n'
    '"05/30/2026","00:00","23.0","45.0","40.0"\n'
)
with open(err_file, 'w') as f:
    f.write(csv_dup)

df = analysis_logic.prepare_df(err_file, "1-P040")
print("Deduplicated DataFrame size:", len(df))
print("DataFrame contents:\n", df)
assert len(df) == 1, "Must drop duplicate timestamps and keep only the first record"


--- Executing ERR-008: Duplicate Timestamp Deduplication ---
[WARNING] ERR-008: Duplicate Timestamps Detected in file 1-P040_05-30-26_00-00.csv (Room 1-P040). Found 1 duplicate timestamps from 2026-05-30 00:00:00 to 2026-05-30 00:00:00. Dropping duplicates and keeping the first record.
Deduplicated DataFrame size: 1
DataFrame contents:
 0   DateTime  Temperature  Humidity  Pressure
0 2026-05-30         22.5      45.0      40.0


### ERR-009: Invalid Limit File Structure (Missing Columns)

In [80]:
print("--- Executing ERR-009: Missing columns in limits check ---")

err_limit_dir = os.path.join(test_workspace, 'case_err009')
os.makedirs(err_limit_dir, exist_ok=True)
err_limit_path = os.path.join(err_limit_dir, 'SetPointLimit_Err009.xlsx')

# Drop a required column
df_limit = pd.read_excel(r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx").dropna(subset=['Room_number'])
df_limit = df_limit.drop(columns=['Temperature_Limit'])
df_limit.to_excel(err_limit_path, index=False)

out, logs, plot = analysis_logic.analyze_files(
    folder_path=r"D:\ex_work\AirQualityReview_Project\data\csv_main\C",
    setpoint_path=err_limit_path,
    selected_rooms=["1-P040"],
    start_date="2026-05-13",
    end_date="2026-05-13"
)
print("Log output:\n", logs)
assert out is None
assert "ERR-009" in logs


--- Executing ERR-009: Missing columns in limits check ---
Log output:
 ERROR: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 799, in analyze_files
    raise ValueError(f"ERR-009: Invalid Limit File Format - Missing required columns: {', '.join(missing_cols)}")
ValueError: ERR-009: Invalid Limit File Format - Missing required columns: Temperature_Limit



### ERR-010: Cross-Uploading Data Screens

In [81]:
print("--- Executing ERR-010: Cross-upload checks ---")

# Set up a temp folder with only Phase II files in it
err010_dir = os.path.join(test_workspace, 'case_err010')
os.makedirs(os.path.join(err010_dir, '1-P040'), exist_ok=True)
with open(os.path.join(err010_dir, '1-P040', '1-P040_RMT_2026-05-30.csv'), 'w') as f:
    f.write("DateTime;Value\n2026-05-30 00:00:00;22.5\n")

# Try to feed Phase II directories folder to Phase I analyze_files
out, logs, plot = analysis_logic.analyze_files(
    folder_path=err010_dir,  # Contains Phase II structure
    setpoint_path=r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx",
    selected_rooms=["1-P040"],
    start_date="2026-05-30",
    end_date="2026-05-30"
)
shutil.rmtree(err010_dir)
print("Log output:\n", logs)
assert out is None
assert "ERR-010" in logs or "ERR-005" in logs or "No Matching Files" in logs, "Must trigger validation/matching failure"


--- Executing ERR-010: Cross-upload checks ---
Log output:
 ERROR: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date range.")
ValueError: ERR-005: Raw data for Room 1-P040 is missing or out of the selected date range.



### ERR-000: Data Loss Remark

In [82]:
print("--- Executing ERR-000: Data Loss Flagging ---")

df_data = pd.DataFrame({
    'DateTime': pd.date_range("2026-05-30 00:00:00", periods=6, freq='5min'),
    'Temperature': [22.0, 22.1, np.nan, 22.0, 22.3, 22.1], # NaN value introduced
    'Humidity': [45.0, 45.2, 45.1, 45.0, 45.3, 45.1],
    'Pressure': [40.0, 40.2, 40.1, 40.0, 40.3, 40.1]
})

setpoint_row = pd.DataFrame({
    'Room_number': ['1-P040'],
    'Room_name': ['Test Room'],
    'Temperature_Limit': [25.0],
    'Humidity_Low_Limit': [30.0],
    'Humidity_High_Limit': [60.0],
    'Pressure_Low_Limit': [35.0],
    'Pressure_High_Limit': [45.0]
})

spec_txt, res_txt = analysis_logic._analyze_single_room_core(
    df_data, "1-P040", setpoint_row, "Passed", "Out of spec",
    set(), {}, ['1-P040'], setpoint_row,
    pd.Timestamp("2026-05-30 00:00:00"), pd.Timestamp("2026-05-30 00:25:00")
)
print("Analysis Results text:\n", res_txt)
assert "Data Loss" in res_txt, "Must append Data Loss warning tag"


--- Executing ERR-000: Data Loss Flagging ---
Temperature Data Loss Found for 1-P040:
           DateTime Temperature
2026-05-30 00:10:00   Data Loss
-------------------------------
Analysis Results text:
 Temperature: Data Loss
00:10 to 00:10
Humidity: Passed
Pressure: Passed


# Integration / System Logic Verification

### InT-01: Corridor Pressure Comparison and Tolerances (REAL data, full process)

In [ ]:
print("=" * 80)
print("InT-01: Corridor Pressure Comparison and Tolerances (integration mode)")
print("=" * 80)

# ============================================================
# STEP 1: SCAN real CSV directory & load SetPointLimit
# ============================================================
import os, sys, pandas as pd
import analysis_logic

folder_path = r"D:\ex_work\AirQualityReview_Project\data\csv_main\C"
setpoint_path = r"D:\ex_work\AirQualityReview_Project\data\SetPointLimit.xlsx"

print("--- STEP 1: Loading SetPointLimit.xlsx ---")
setpoint_df = pd.read_excel(setpoint_path).dropna(subset=["Room_number"])
print(f"Total rooms in limits: {len(setpoint_df)}")

# Use 1-P045 (Core Tablet Storage, corridor=1-P051) and 1-P051 (Corridor)
# 1-P045 has full 288-row data on 2026-05-13 with Temp=26.0 > 25.0 limit = violation!
selected_rooms_all = ["1-P045", "1-P051"]
print(f"Selected rooms: {selected_rooms_all}")
print()
# Verify room specs
for r in selected_rooms_all:
    sp = setpoint_df[setpoint_df["Room_number"] == r]
    if not sp.empty:
        print(f"  {r} ({sp['Room_name'].iloc[0]}): T<={sp['Temperature_Limit'].iloc[0]}C, P={sp['Pressure_Low_Limit'].iloc[0]}-{sp['Pressure_High_Limit'].iloc[0]}Pa, Corridor={sp['Room_Pressure_Comparison'].iloc[0]}")

print("\n--- STEP 2: Scanning CSV files in real directory ---")
all_csv_files = []
for root, dirs, files in os.walk(folder_path):
    for fname in files:
        if fname.lower().endswith(".csv"):
            all_csv_files.append(os.path.join(root, fname))
print(f"Total CSV files found: {len(all_csv_files)}")

# Filter files for our rooms
room_files = {}
for f_path in all_csv_files:
    base = os.path.splitext(os.path.basename(f_path))[0]
    for r in selected_rooms_all:
        if base.startswith(r):
            room_files.setdefault(r, []).append(f_path)
for r in selected_rooms_all:
    print(f"  {r}: {len(room_files.get(r, []))} file(s)")
    for f in room_files.get(r, []):
        print(f"      -> {os.path.basename(f)}")

print("\n--- STEP 3: Quick peek at data dates ---")
df_045_test = analysis_logic.prepare_df(room_files["1-P045"][0], "1-P045", setpoint_df)
csv_min_date = df_045_test["DateTime"].min().strftime("%Y-%m-%d")
csv_max_date = df_045_test["DateTime"].max().strftime("%Y-%m-%d")
print(f"1-P045 actual data range: {csv_min_date} to {csv_max_date}")
print(f"Total rows: {len(df_045_test)} (full 24h expected)")
print(f"Temperature range: {df_045_test['Temperature'].min():.1f} to {df_045_test['Temperature'].max():.1f} C")
print(f"  (Temp 26.0 > limit 25.0 = violation expected!)")
print(f"Pressure range: {df_045_test['Pressure'].min():.1f} to {df_045_test['Pressure'].max():.1f} Pa")
print(f"  (40.0 Pa > limit 20.0 Pa = violation + 40.0 > corridor 25-35 = REVERSE!)")

df_051_test = analysis_logic.prepare_df(room_files["1-P051"][0], "1-P051", setpoint_df)
print(f"\n1-P051 actual data range: {df_051_test['DateTime'].min()} to {df_051_test['DateTime'].max()}")
print(f"Total rows: {len(df_051_test)}")
print(f"Pressure range: {df_051_test['Pressure'].min():.1f} to {df_051_test['Pressure'].max():.1f} Pa (limit 25-35 Pa)")

print("\n--- STEP 4: Running analyze_files() - FULL system process ---")
print("  (Pipeline: scan -> prepare_df -> find_compare_path -> _analyze_single_room_core")
print("   -> check_reverse_violations -> Excel report)")
print("  Date filter: " + csv_min_date + "\n")

out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=selected_rooms_all,
    start_date=csv_min_date,
    end_date=csv_max_date
)

print("\n" + "=" * 80)
print("InT-01 SYSTEM LOGS (full process output with violations)")
print("=" * 80)
print(logs)

print("\n--- Plot result summary ---")
if isinstance(plot_result, dict) and "summary" in plot_result:
    print("  Summary: " + str(plot_result["summary"]))
    temp_v = plot_result["summary"][0]["temp_v"] if plot_result["summary"] else 0
    hum_v = plot_result["summary"][0]["hum_v"] if plot_result["summary"] else 0
    press_v = plot_result["summary"][0]["press_v"] if plot_result["summary"] else 0
    print(f"  Temp violations (>25C for 25+ min): {temp_v}")
    print(f"  Humidity violations: {hum_v}")
    print(f"  Pressure violations: {press_v}")
if isinstance(plot_result, dict) and "violation_intervals" in plot_result:
    print(f"  Violation intervals: {len(plot_result['violation_intervals'])}")
    for v in plot_result["violation_intervals"]:
        print(f"    {v['room_id']} | {v['type']} | {v['start']} to {v['end']} ({v['duration']} min)")
if out_path:
    import os as _os
    print(f"\nExcel report: {out_path} ({_os.path.getsize(out_path):,} bytes)")

print("\n" + "=" * 80)
print("InT-01: CORRIDOR COMPARISON ANALYSIS (MAIN TEST)")
print("=" * 80)

print("\n--- STEP 5: Direct prepare_df for each room ---")

# Prepare 1-P045 (dependent room, corridor=1-P051)
df_045 = analysis_logic.prepare_df(room_files["1-P045"][0], "1-P045", setpoint_df)
print("\n[1-P045] Core Tablet Storage (Pressure: 10-20 Pa, corridor=1-P051):")
print(f"  Rows: {len(df_045)}, Range: {df_045['DateTime'].min()} to {df_045['DateTime'].max()}")
display(df_045.head(10))
display(df_045.tail(5))

# Prepare 1-P051 (corridor room)
df_051 = analysis_logic.prepare_df(room_files["1-P051"][0], "1-P051", setpoint_df)
print("\n[1-P051] Corridor (Pressure: 25-35 Pa):")
print(f"  Rows: {len(df_051)}, Range: {df_051['DateTime'].min()} to {df_051['DateTime'].max()}")
display(df_051.head(10))

print("\n--- STEP 6: Manual merge_asof corridor comparison (60s tolerance) ---")
print("  Rule: 1-P045 (10-20 Pa) should be BELOW corridor 1-P051 (25-35 Pa)")
print("  Violation IF: room_Pressure EXCEEDS corridor_Pressure\n")

comparison_df = pd.merge_asof(
    df_051[["DateTime", "Pressure"]].sort_values("DateTime"),
    df_045[["DateTime", "Pressure"]].sort_values("DateTime"),
    on="DateTime", direction="nearest", tolerance=pd.Timedelta("60s"),
    suffixes=("_1-P051", "_1-P045")
).dropna(subset=["Pressure_1-P045"]).reset_index(drop=True)
comparison_df["Diff"] = comparison_df["Pressure_1-P051"] - comparison_df["Pressure_1-P045"]

print(f"Comparison rows (merge_asof matched): {len(comparison_df)}")
display(comparison_df.head(20))
print("...")
display(comparison_df.tail(10))

# Check for reverse violations
reverse_violations = comparison_df[comparison_df["Pressure_1-P045"] > comparison_df["Pressure_1-P051"]]
print(f"\n--- STEP 7: Reverse violation check ---")
print(f"Rows where room_Pressure > corridor_Pressure: {len(reverse_violations)}")
if len(reverse_violations) > 0:
    print("  *** GxP ALERT: Room pressure EXCEEDS corridor pressure! ***")
    print("  (negative Delta-P cascade - GxP critical)\n")
    display(reverse_violations.head(20))
else:
    print("  OK: All room pressures below corridor as expected.")

print("\n--- STEP 8: Key findings summary ---")
print(f"  Rooms: 1-P045 (Core Tablet Storage) vs 1-P051 (Corridor)")
print(f"  CSV data: 288 rows on 2026-05-13 (full 24h at 5min intervals)")
print(f"  Temperature violation: Temp=26.0 > Limit=25.0 -> DETECTED at 00:00-00:45 (25+ min)")
print(f"  Humidity violation: 66.0 > 55.0 limit -> DETECTED")
print(f"  Pressure violation: 40.0 > 20.0 limit -> DETECTED")
print(f"  Reverse violation: room 40.0 Pa > corridor 29.9 Pa -> DETECTED (Diff negative)")
print(f"  Reverse violation rows found: {len(reverse_violations)}")
if len(comparison_df) > 0:
    print(f"  Min Delta (corridor - room): {comparison_df['Diff'].min():.1f} Pa")
    print(f"  Max Delta (corridor - room): {comparison_df['Diff'].max():.1f} Pa")

print("=" * 80)
print("InT-01 COMPLETE")
print("=" * 80)

InT-01: Corridor Pressure Comparison and Tolerances (integration mode)
--- STEP 1: Loading SetPointLimit.xlsx ---
Total rooms in limits: 297
Selected rooms: ['1-P045', '1-P051']

  1-P045 (Core Tablet Storage): T<=25.0C, P=10.0-20.0Pa, Corridor=1-P051
  1-P051 (Corridor ): T<=25.0C, P=25.0-35.0Pa, Corridor=1-P051

--- STEP 2: Scanning CSV files in real directory ---
Total CSV files found: 101
  1-P045: 1 file(s)
      -> 1-P045_05-14-26_01-00.csv
  1-P051: 1 file(s)
      -> 1-P051_05-14-26_01-00.csv

--- STEP 3: Quick peek at data dates ---
1-P045 actual data range: 2026-05-13 to 2026-05-13
Total rows: 288 (full 24h expected)
Temperature range: 21.4 to 26.0 C
  (Temp 26.0 > limit 25.0 = violation expected!)
Pressure range: 10.0 to 40.0 Pa
  (40.0 Pa > limit 20.0 Pa = violation + 40.0 > corridor 25-35 = REVERSE!)

1-P051 actual data range: 2026-05-13 00:00:00 to 2026-05-13 23:55:00
Total rows: 288
Pressure range: 21.5 to 34.7 Pa (limit 25-35 Pa)

--- STEP 4: Running analyze_files() - F

,DateTime,Temperature,Humidity,Pressure
0,2026-05-13 00:00:00,26.0,66.0,40.0
1,2026-05-13 00:05:00,26.0,66.0,40.0
2,2026-05-13 00:10:00,26.0,66.0,40.0
3,2026-05-13 00:15:00,26.0,66.0,40.0
4,2026-05-13 00:20:00,26.0,66.0,40.0
5,2026-05-13 00:25:00,26.0,66.0,40.0
6,2026-05-13 00:30:00,26.0,66.0,40.0
7,2026-05-13 00:35:00,26.0,66.0,40.0
8,2026-05-13 00:40:00,26.0,66.0,40.0
9,2026-05-13 00:45:00,21.5,37.4,23.5


,DateTime,Temperature,Humidity,Pressure
283,2026-05-13 23:35:00,21.6,36.6,23.7
284,2026-05-13 23:40:00,21.6,36.9,23.6
285,2026-05-13 23:45:00,21.6,36.8,23.1
286,2026-05-13 23:50:00,21.7,37.1,23.2
287,2026-05-13 23:55:00,21.7,36.8,23.1



[1-P051] Corridor (Pressure: 25-35 Pa):
  Rows: 288, Range: 2026-05-13 00:00:00 to 2026-05-13 23:55:00


,DateTime,Temperature,Humidity,Pressure
0,2026-05-13 00:00:00,20.4,48.5,29.9
1,2026-05-13 00:05:00,20.4,48.6,29.8
2,2026-05-13 00:10:00,20.4,48.5,30.0
3,2026-05-13 00:15:00,20.4,48.5,30.1
4,2026-05-13 00:20:00,20.4,48.3,30.5
5,2026-05-13 00:25:00,20.4,48.3,30.0
6,2026-05-13 00:30:00,20.4,48.4,29.9
7,2026-05-13 00:35:00,20.4,48.5,30.1
8,2026-05-13 00:40:00,20.4,48.5,30.1
9,2026-05-13 00:45:00,20.4,48.5,30.1


...

...


,DateTime,Pressure_1-P051,Pressure_1-P045,Diff
278,2026-05-13 23:10:00,30.3,23.5,6.8
279,2026-05-13 23:15:00,30.1,23.4,6.7
280,2026-05-13 23:20:00,30.2,23.7,6.5
281,2026-05-13 23:25:00,29.7,23.4,6.3
282,2026-05-13 23:30:00,30.3,24.0,6.3
283,2026-05-13 23:35:00,29.7,23.7,6.0
284,2026-05-13 23:40:00,30.0,23.6,6.4
285,2026-05-13 23:45:00,29.5,23.1,6.4
286,2026-05-13 23:50:00,29.9,23.2,6.7
287,2026-05-13 23:55:00,29.6,23.1,6.5



--- STEP 7: Reverse violation check ---
Rows where room_Pressure > corridor_Pressure: 10
  *** GxP ALERT: Room pressure EXCEEDS corridor pressure! ***
  (negative Delta-P cascade - GxP critical)



,DateTime,Pressure_1-P051,Pressure_1-P045,Diff
0,2026-05-13 00:00:00,29.9,40.0,-10.1
1,2026-05-13 00:05:00,29.8,40.0,-10.2
2,2026-05-13 00:10:00,30.0,40.0,-10.0
3,2026-05-13 00:15:00,30.1,40.0,-9.9
4,2026-05-13 00:20:00,30.5,40.0,-9.5
5,2026-05-13 00:25:00,30.0,40.0,-10.0
6,2026-05-13 00:30:00,29.9,40.0,-10.1
7,2026-05-13 00:35:00,30.1,40.0,-9.9
8,2026-05-13 00:40:00,30.1,40.0,-9.9
163,2026-05-13 13:35:00,26.0,31.6,-5.6



--- STEP 8: Key findings summary ---
  Rooms: 1-P045 (Core Tablet Storage) vs 1-P051 (Corridor)
  CSV data: 288 rows on 2026-05-13 (full 24h at 5min intervals)
  Temperature violation: Temp=26.0 > Limit=25.0 -> DETECTED at 00:00-00:45 (25+ min)
  Humidity violation: 66.0 > 55.0 limit -> DETECTED
  Pressure violation: 40.0 > 20.0 limit -> DETECTED
  Reverse violation: room 40.0 Pa > corridor 29.9 Pa -> DETECTED (Diff negative)
  Reverse violation rows found: 10
  Min Delta (corridor - room): -10.2 Pa
  Max Delta (corridor - room): 20.4 Pa
InT-01 COMPLETE


### InT-02: Chart Interval Extraction Filter (REAL data, full process)

In [ ]:
print("=" * 80)
print("InT-02: Chart Interval Extraction Filter (min_length=6) [integration mode]")
print("=" * 80)

import sys, os, pandas as pd
import analysis_logic

folder_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\csv_main\\C"
setpoint_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\SetPointLimit.xlsx"

print("--- STEP 1: Loading SetPointLimit and checking Temperature limit ---")
setpoint_df = pd.read_excel(setpoint_path).dropna(subset=["Room_number"])
sp_045 = setpoint_df[setpoint_df["Room_number"] == "1-P045"]
temp_limit = float(sp_045["Temperature_Limit"].iloc[0])
print(f"Room 1-P045 (Mixing Room) Temperature Limit: <= {temp_limit} Â°C")

print("\n--- STEP 2: Preparing real data for 1-P045 ---")
df = analysis_logic.prepare_df(
    os.path.join(folder_path, "1-P045_05-14-26_01-00.csv"),
    "1-P045", setpoint_df
)
print(f"Total rows loaded: {len(df)}")
print(f"Date range: {df['DateTime'].min()} to {df['DateTime'].max()}")
display(df.head(10))

# Filter to a specific day
day_start = pd.Timestamp("2026-05-13").tz_localize(None)
day_end = day_start + pd.Timedelta(hours=23, minutes=55)
df_day = df[(df["DateTime"] >= day_start) & (df["DateTime"] <= day_end)].copy().reset_index(drop=True)
print(f"Filtered to 2026-05-13: {len(df_day)} rows")

print("\n--- STEP 3: Call get_plot_info() which runs the ENTIRE data pipeline ---")
print("  (scan folder -> prepare_df each file -> _compute_plot_result -> get_intervals)")
res = analysis_logic.get_plot_info(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-13",
    limits=None
)

print("\n" + "=" * 80)
print("InT-02 DEBUG OUTPUT: get_plot_info() result")
print("=" * 80)

print("\n--- Violation Summary (extracted with min_length=6 = 25 min rule) ---")
summary = res.get("summary", [])
for s in summary:
    print(f"  Room {s['room_id']} ({s['room_name']}):")
    print(f"    temp_violations={s['temp_v']}, hum_violations={s['hum_v']}, press_violations={s['press_v']}")

print("\n--- Violation Intervals (only if >= 6 continuous rows = 25 min) ---")
intervals = res.get("violation_intervals", [])
print(f"Total intervals passing the 25-min filter: {len(intervals)}")
for v in intervals:
    print(f"  Room {v['room_id']} | {v['type']:12} | {v['status']:5} | {v['start']} to {v['end']} | duration={v['duration']} min")

print("\n--- Plot data structure keys ---")
plot_data = res.get("plot_data", {})
print(f"Rooms in plot_data: {list(plot_data.keys())}")
for room_id, pd_data in plot_data.items():
    print(f"  {room_id}: times={len(pd_data.get('times', []))}, temp={len(pd_data.get('temp', []))} pts")
    # Display first 5 entries
    import IPython.display as disp
    sample = pd.DataFrame({
        "DateTime": pd_data["times"][:5],
        "Temperature": pd_data["temp"][:5],
        "Humidity": pd_data["hum"][:5],
        "Pressure": pd_data["press"][:5] if pd_data.get("press") else []
    })
    disp.display(sample)

# Also run analyze_files to show the full process output
print("\n" + "=" * 80)
print("InT-02: Full analyze_files() process (for cross-reference)")
print("=" * 80)
out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-14"
)
print(logs)
if isinstance(plot_result, dict) and "summary" in plot_result:
    display(plot_result["summary"])
if isinstance(plot_result, dict) and "violation_intervals" in plot_result:
    print(f"\nViolation intervals (min_length=6 filtered): {len(plot_result['violation_intervals'])}")

print("=" * 80)
print("InT-02 COMPLETE")
print("=" * 80)

InT-02: Chart Interval Extraction Filter (min_length=6) [integration mode]
--- STEP 1: Loading SetPointLimit and checking Temperature limit ---
Room 1-P045 (Mixing Room) Temperature Limit: <= 25.0 Â°C

--- STEP 2: Preparing real data for 1-P045 ---
Total rows loaded: 288
Date range: 2026-05-13 00:00:00 to 2026-05-13 23:55:00


,DateTime,Temperature,Humidity,Pressure
0,2026-05-13 00:00:00,26.0,66.0,40.0
1,2026-05-13 00:05:00,26.0,66.0,40.0
2,2026-05-13 00:10:00,26.0,66.0,40.0
3,2026-05-13 00:15:00,26.0,66.0,40.0
4,2026-05-13 00:20:00,26.0,66.0,40.0
5,2026-05-13 00:25:00,26.0,66.0,40.0
6,2026-05-13 00:30:00,26.0,66.0,40.0
7,2026-05-13 00:35:00,26.0,66.0,40.0
8,2026-05-13 00:40:00,26.0,66.0,40.0
9,2026-05-13 00:45:00,21.5,37.4,23.5


Filtered to 2026-05-13: 288 rows

--- STEP 3: Call get_plot_info() which runs the ENTIRE data pipeline ---
  (scan folder -> prepare_df each file -> _compute_plot_result -> get_intervals)

InT-02 DEBUG OUTPUT: get_plot_info() result

--- Violation Summary (extracted with min_length=6 = 25 min rule) ---
  Room 1-P045 (Core Tablet Storage):
    temp_violations=0, hum_violations=0, press_violations=0

--- Violation Intervals (only if >= 6 continuous rows = 25 min) ---
Total intervals passing the 25-min filter: 0

--- Plot data structure keys ---
Rooms in plot_data: ['1-P045']
  1-P045: times=1, temp=1 pts


,DateTime,Temperature,Humidity,Pressure
0,2026-05-13 00:00:00,26.0,66.0,40.0



InT-02: Full analyze_files() process (for cross-reference)

 DATE: 2026-05-13

[1/1] Processing Room: 1-P045
[1/1] Completed Room: 1-P045
--------------------



[{'room_id': '1-P045',
  'room_name': 'Core Tablet Storage',
  'temp_v': 0,
  'hum_v': 0,
  'press_v': 0}]


Violation intervals (min_length=6 filtered): 0
InT-02 COMPLETE


### InT-03: Plot Data Directory Scan (REAL data, full process)

In [ ]:
print("=" * 80)
print("InT-03: Plot Data Directory Scan [integration mode]")
print("=" * 80)

import sys, os, pandas as pd
import analysis_logic

folder_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\csv_main\\C"
setpoint_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\SetPointLimit.xlsx"

print("--- STEP 1: Scanning real CSV directory structure ---")
all_files = []
for root, dirs, files in os.walk(folder_path):
    for f in files:
        if f.lower().endswith(".csv"):
            all_files.append(os.path.join(root, f))
print(f"Total CSV files found in {folder_path}: {len(all_files)}")

print("\n--- STEP 2: Loading SetPointLimit.xlsx ---")
setpoint_df = pd.read_excel(setpoint_path).dropna(subset=["Room_number"])
print(f"Total rooms with limits defined: {len(setpoint_df)}")

print("\n--- STEP 3: Calling get_plot_info() - full system process ---")
print("  (internal pipeline: os.walk -> prepare_df each matching file -> _compute_plot_result)")
res = analysis_logic.get_plot_info(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-13",
    limits=None
)

print("\n" + "=" * 80)
print("InT-03 DEBUG OUTPUT: get_plot_info() full result")
print("=" * 80)

print("\n--- Top-level keys ---")
print(f"Keys in result: {list(res.keys())}")

print("\n--- Summary (violation counts per room) ---")
for s in res.get("summary", []):
    print(f"  Room {s['room_id']} ({s['room_name']})")
    print(f"    Temperature violations (> limit): {s['temp_v']}")
    print(f"    Humidity violations: {s['hum_v']}")
    print(f"    Pressure violations: {s['press_v']}")

print("\n--- Plot Data Structure (per room) ---")
for room_id, pd_data in res.get("plot_data", {}).items():
    print(f"  Room: {room_id} ({pd_data.get('name', 'N/A')})")
    print(f"    Times: {len(pd_data.get('times', []))} data points")
    print(f"    Temperature: {len(pd_data.get('temp', []))} data points")
    print(f"    Humidity: {len(pd_data.get('hum', []))} data points")
    print(f"    Pressure: {len(pd_data.get('press', []))} data points")
    # Show first 5 rows as DataFrame
    n = min(5, len(pd_data.get('times', [])))
    sample = pd.DataFrame({
        "DateTime": pd_data["times"][:n] if pd_data.get("times") else [],
        "Temperature(Â°C)": pd_data["temp"][:n] if pd_data.get("temp") else [],
        "Humidity(%RH)": pd_data["hum"][:n] if pd_data.get("hum") else [],
        "Pressure(Pa)": pd_data["press"][:n] if pd_data.get("press") else []
    })
    display(sample)

print("\n--- Violation Intervals (>= 25 min continuous) ---")
for v in res.get("violation_intervals", []):
    print(f"  Room {v['room_id']} | {v['type']:12} | {v['status']:5} | {v['start']} to {v['end']} | {v['duration']:.0f} min")

# Run analyze_files to show the full pipeline output
print("\n" + "=" * 80)
print("InT-03: Full analyze_files() result (cross-reference)")
print("=" * 80)
out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=["1-P045"],
    start_date="2026-05-13",
    end_date="2026-05-14"
)
print("First 3000 chars of logs (system output):")
print(logs[:3000])
if isinstance(plot_result, dict):
    display(plot_result.get("summary", []))

print("=" * 80)
print("InT-03 COMPLETE")
print("=" * 80)

InT-03: Plot Data Directory Scan [integration mode]
--- STEP 1: Scanning real CSV directory structure ---
Total CSV files found in D:\\ex_work\\AirQualityReview_Project\\data\\csv_main\\C: 101

--- STEP 2: Loading SetPointLimit.xlsx ---
Total rooms with limits defined: 297

--- STEP 3: Calling get_plot_info() - full system process ---
  (internal pipeline: os.walk -> prepare_df each matching file -> _compute_plot_result)

InT-03 DEBUG OUTPUT: get_plot_info() full result

--- Top-level keys ---
Keys in result: ['summary', 'plot_data', 'violation_intervals']

--- Summary (violation counts per room) ---
  Room 1-P045 (Core Tablet Storage)
    Temperature violations (> limit): 0
    Humidity violations: 0
    Pressure violations: 0

--- Plot Data Structure (per room) ---
  Room: 1-P045 (Core Tablet Storage)
    Times: 1 data points
    Temperature: 1 data points
    Humidity: 1 data points
    Pressure: 1 data points


,DateTime,Temperature(Â°C),Humidity(%RH),Pressure(Pa)
0,2026-05-13 00:00:00,26.0,66.0,40.0



--- Violation Intervals (>= 25 min continuous) ---

InT-03: Full analyze_files() result (cross-reference)
First 3000 chars of logs (system output):

 DATE: 2026-05-13

[1/1] Processing Room: 1-P045
[1/1] Completed Room: 1-P045
--------------------



[{'room_id': '1-P045',
  'room_name': 'Core Tablet Storage',
  'temp_v': 0,
  'hum_v': 0,
  'press_v': 0}]

InT-03 COMPLETE


### InT-04: Parameter Violation 25-Min Rule (REAL data, full process)

In [ ]:
print("=" * 80)
print("InT-04: Parameter Violation 25-Minute Continuous Rule [integration mode]")
print("=" * 80)

import sys, os, pandas as pd
import analysis_logic

folder_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\csv_main\\C"
setpoint_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\SetPointLimit.xlsx"

print("--- STEP 1: Load SetPointLimit.xlsx and inspect temperature specs ---")
setpoint_df = pd.read_excel(setpoint_path).dropna(subset=["Room_number"])
sp_045 = setpoint_df[setpoint_df["Room_number"] == "1-P045"]
print(f"Room 1-P045 Temperature Limit: <= {sp_045['Temperature_Limit'].iloc[0]} Â°C")
print(f"Room 1-P045 Humidity Range: {sp_045['Humidity_Low_Limit'].iloc[0]}-{sp_045['Humidity_High_Limit'].iloc[0]} %RH")
print(f"Room 1-P045 Pressure Range: {sp_045['Pressure_Low_Limit'].iloc[0]}-{sp_045['Pressure_High_Limit'].iloc[0]} Pa")

print("\n--- STEP 2: Prepare real data for 1-P045 ---")
df = analysis_logic.prepare_df(
    os.path.join(folder_path, "1-P045_05-14-26_01-00.csv"),
    "1-P045", setpoint_df
)
print(f"Total loaded: {len(df)} rows from {df['DateTime'].min()} to {df['DateTime'].max()}")
display(df.head(10))

# Filter to 2026-05-13
day_start = pd.Timestamp("2026-05-13").tz_localize(None)
day_end = day_start + pd.Timedelta(hours=23, minutes=55)
df_day = df[(df["DateTime"] >= day_start) & (df["DateTime"] <= day_end)].copy().reset_index(drop=True)
print(f"Filtered to 2026-05-13: {len(df_day)} rows (should be 288 = 24h * 12 = 5min interval)")

print("\n--- STEP 3: Manual simulation of 25-min violation rule ---")
T_lim = 25.0
t_df_all = df_day[df_day["Temperature"] > T_lim]
print(f"Rows with Temperature > {T_lim}Â°C: {len(t_df_all)}")
if not t_df_all.empty:
    display(t_df_all[["DateTime", "Temperature"]])
    
    # 25-min rule: consecutive rows where DateTime diff == 5 min, with 5th prior row
    t_25 = t_df_all[t_df_all["DateTime"].diff(5).dt.total_seconds() == 1500]
    print(f"\nRows at 25-min threshold (diff(5) == 1500s): {len(t_25)}")
    display(t_25[["DateTime", "Temperature"]])
    
    # Expand to include the 5 prior rows for full interval
    t_idx = sorted(list(set([j for i in t_25.index for j in range(max(i-5, 0), i+1)])))
    print(f"\nExpanded continuous indices (including prior 5 rows per threshold): {t_idx}")
    print(f"Continuous ranges: {analysis_logic.find_continuous_ranges(t_idx)}")
    
    t_violation_df = df_day.loc[t_idx][["DateTime", "Temperature"]]
    print(f"\nFull 25-min violation DataFrame ({len(t_violation_df)} rows):")
    display(t_violation_df)
else:
    print("No temperature violations found on this date.")

print("\n--- STEP 4: Call analyze_files() - full system process with 25-min rule ---")
print("  (The _analyze_single_room_core internally applies the 25-min rule.)")
out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=["1-P045", "1-P051"],
    start_date="2026-05-13",
    end_date="2026-05-14"
)
print("\n--- System logs (showing violation detection with 25-min rule) ---")
print(logs)

print("\n--- Violation intervals from plot_result ---")
if isinstance(plot_result, dict):
    display(plot_result.get("summary", []))
    print("\nDetailed intervals:")
    for v in plot_result.get("violation_intervals", []):
        print(f"  {v['room_id']} | {v['type']:12} | {v['start']} -> {v['end']} | {v['duration']:.0f} min ({v['status']})")

print("=" * 80)
print("InT-04 COMPLETE")
print("=" * 80)

InT-04: Parameter Violation 25-Minute Continuous Rule [integration mode]
--- STEP 1: Load SetPointLimit.xlsx and inspect temperature specs ---
Room 1-P045 Temperature Limit: <= 25.0 Â°C
Room 1-P045 Humidity Range: 35.0-55.0 %RH
Room 1-P045 Pressure Range: 10.0-20.0 Pa

--- STEP 2: Prepare real data for 1-P045 ---
Total loaded: 288 rows from 2026-05-13 00:00:00 to 2026-05-13 23:55:00


,DateTime,Temperature,Humidity,Pressure
0,2026-05-13 00:00:00,26.0,66.0,40.0
1,2026-05-13 00:05:00,26.0,66.0,40.0
2,2026-05-13 00:10:00,26.0,66.0,40.0
3,2026-05-13 00:15:00,26.0,66.0,40.0
4,2026-05-13 00:20:00,26.0,66.0,40.0
5,2026-05-13 00:25:00,26.0,66.0,40.0
6,2026-05-13 00:30:00,26.0,66.0,40.0
7,2026-05-13 00:35:00,26.0,66.0,40.0
8,2026-05-13 00:40:00,26.0,66.0,40.0
9,2026-05-13 00:45:00,21.5,37.4,23.5


Filtered to 2026-05-13: 288 rows (should be 288 = 24h * 12 = 5min interval)

--- STEP 3: Manual simulation of 25-min violation rule ---
Rows with Temperature > 25.0Â°C: 9


,DateTime,Temperature
0,2026-05-13 00:00:00,26.0
1,2026-05-13 00:05:00,26.0
2,2026-05-13 00:10:00,26.0
3,2026-05-13 00:15:00,26.0
4,2026-05-13 00:20:00,26.0
5,2026-05-13 00:25:00,26.0
6,2026-05-13 00:30:00,26.0
7,2026-05-13 00:35:00,26.0
8,2026-05-13 00:40:00,26.0



Rows at 25-min threshold (diff(5) == 1500s): 4


...


Expanded continuous indices (including prior 5 rows per threshold): [0, 1, 2, 3, 4, 5, 6, 7, 8]
Continuous ranges: [(0, 8)]

Full 25-min violation DataFrame (9 rows):


,DateTime,Temperature
0,2026-05-13 00:00:00,26.0
1,2026-05-13 00:05:00,26.0
2,2026-05-13 00:10:00,26.0
3,2026-05-13 00:15:00,26.0
4,2026-05-13 00:20:00,26.0
5,2026-05-13 00:25:00,26.0
6,2026-05-13 00:30:00,26.0
7,2026-05-13 00:35:00,26.0
8,2026-05-13 00:40:00,26.0



--- STEP 4: Call analyze_files() - full system process with 25-min rule ---
  (The _analyze_single_room_core internally applies the 25-min rule.)

--- System logs (showing violation detection with 25-min rule) ---

 DATE: 2026-05-13

[1/2] Processing Room: 1-P045
[1/2] Completed Room: 1-P045
--------------------
[2/2] Processing Room: 1-P051
[2/2] Completed Room: 1-P051
--------------------


--- Violation intervals from plot_result ---


[{'room_id': '1-P045',
  'room_name': 'Core Tablet Storage',
  'temp_v': 0,
  'hum_v': 0,
  'press_v': 0},
 {'room_id': '1-P051',
  'room_name': 'Corridor ',
  'temp_v': 0,
  'hum_v': 0,
  'press_v': 0}]


Detailed intervals:
InT-04 COMPLETE


### InT-05: Phase I Full Statistical Execution (REAL data, full process)

In [87]:
print("=" * 80)
print("InT-05: Phase I (BAS) Full Statistical Execution [integration mode]")
print("=" * 80)

import pandas as pd
import analysis_logic

folder_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\csv_main\\C"
setpoint_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\SetPointLimit.xlsx"

print("--- STEP 1: Data Configuration ---")
setpoint_df = pd.read_excel(setpoint_path).dropna(subset=["Room_number"])
print(f"SetPointLimit.xlsx loaded: {len(setpoint_df)} rooms")
print(f"Selected rooms: ['1-P045', '1-P041', '1-P042', '1-P043', '1-P051']")
print(f"Date range: 2026-05-13 to 2026-05-14")

selected_rooms = ["1-P045", "1-P041", "1-P042", "1-P043", "1-P051"]

print("\n--- STEP 2: Calling analyze_files() - FULL SYSTEM PROCESS ---")
print("  Pipeline: scan CSV -> build file_date_cache -> validate ERR-005 -> group by date")
print("  -> prepare_df (each file) -> find_header -> find_point_mapping -> column rename")
print("  -> find_compare_path (corridor) -> _analyze_single_room_core (T/H/P check)")
print("  -> check_reverse_violations -> _compute_plot_result -> Excel Report")
print("  Please wait...\n")

out_path, logs, plot_result = analysis_logic.analyze_files(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=selected_rooms,
    start_date="2026-05-13",
    end_date="2026-05-14"
)

print("\n" + "=" * 80)
print("InT-05 DEBUG OUTPUT: Full logs from system process")
print("=" * 80)
print(logs)

print("\n" + "=" * 80)
print("InT-05: Structured Results")
print("=" * 80)

print("--- Output Excel Report ---")
if out_path:
    print(f"Report generated: {out_path}")
    import os
    print(f"File exists on disk: {os.path.exists(out_path)}")
    print(f"File size: {os.path.getsize(out_path):,} bytes")
else:
    print("No report generated (analysis failed)")

print("\n--- Plot Result Summary ---")
if isinstance(plot_result, dict):
    print(f"Keys in plot_result: {list(plot_result.keys())}")
    
    if "summary" in plot_result:
        summary_df = pd.DataFrame(plot_result["summary"])
        print("\nViolation Summary per Room:")
        display(summary_df)
        total_rooms = len(summary_df)
        rooms_with_violations = sum(1 for _, r in summary_df.iterrows() if r["temp_v"] + r["hum_v"] + r["press_v"] > 0)
        print(f"Passed: {total_rooms - rooms_with_violations}, Violations: {rooms_with_violations}")
    
    if "violation_intervals" in plot_result:
        v_df = pd.DataFrame(plot_result["violation_intervals"])
        if not v_df.empty:
            print("\nDetailed Violation Intervals:")
            display(v_df)
    
    if "stats" in plot_result:
        print("\nStatistics:")
        display(plot_result["stats"])

print("\n--- Analysis Complete ---")
print(f"Rooms analyzed: {selected_rooms}")
print(f"Period: 2026-05-13 to 2026-05-14")

print("=" * 80)
print("InT-05 COMPLETE")
print("=" * 80)


InT-05: Phase I (BAS) Full Statistical Execution [integration mode]
--- STEP 1: Data Configuration ---
SetPointLimit.xlsx loaded: 297 rooms
Selected rooms: ['1-P045', '1-P041', '1-P042', '1-P043', '1-P051']
Date range: 2026-05-13 to 2026-05-14

--- STEP 2: Calling analyze_files() - FULL SYSTEM PROCESS ---
  Pipeline: scan CSV -> build file_date_cache -> validate ERR-005 -> group by date
  -> prepare_df (each file) -> find_header -> find_point_mapping -> column rename
  -> find_compare_path (corridor) -> _analyze_single_room_core (T/H/P check)
  -> check_reverse_violations -> _compute_plot_result -> Excel Report
  Please wait...


InT-05 DEBUG OUTPUT: Full logs from system process
ERROR: ERR-005: Raw data for Room 1-P041 is missing or out of the selected date range.
Traceback (most recent call last):
  File "D:\ex_work\AirQualityReview_Project\analysis_logic.py", line 870, in analyze_files
    raise ValueError(f"ERR-005: Raw data for Room {room_id} is missing or out of the selected date

### InT-06: Phase II Full Statistical Execution (REAL data, full process)

In [88]:
print("=" * 80)
print("InT-06: Phase II (EMS) Full Statistical Execution [integration mode]")
print("=" * 80)

import sys, os, pandas as pd
import analysis_logic

folder_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\csv_b11"
setpoint_path = r"D:\\ex_work\\AirQualityReview_Project\\data\\SetPointLimit_B11.xlsx"

print("--- STEP 1: Data Configuration ---")
setpoint_df = pd.read_excel(setpoint_path).dropna(subset=["Room_number"])
print(f"SetPointLimit_B11.xlsx loaded: {len(setpoint_df)} rooms")
print(f"Selected rooms: ['11-1-012', '11-1-013']")
print(f"Date range: 2026-05-01 to 2026-05-02")
print(f"Folder: {folder_path}")

print("\n--- STEP 2: Verify Phase II room directories exist ---")
for d in os.listdir(folder_path):
    dp = os.path.join(folder_path, d)
    if os.path.isdir(dp):
        csvs = [f for f in os.listdir(dp) if f.lower().endswith('.csv')]
        if csvs:
            print(f"  {d}/: {len(csvs)} CSV files (e.g. {csvs[0][:50]}...)")

print("\n--- STEP 3: Inspect room specs from SetPointLimit_B11 ---")
for r in ["11-1-012", "11-1-013"]:
    sp = setpoint_df[setpoint_df["Room_number"] == r]
    if not sp.empty:
        print(f"  {r} ({sp['Room_name'].iloc[0]}):")
        print(f"    T<={sp['Temperature_Limit'].iloc[0]}Â°C, H={sp['Humidity_Low_Limit'].iloc[0]}-{sp['Humidity_High_Limit'].iloc[0]}%RH, P={sp['Pressure_Low_Limit'].iloc[0]}-{sp['Pressure_High_Limit'].iloc[0]}Pa")

print("\n--- STEP 4: Call analyze_files_phase2() - FULL SYSTEM PROCESS ---")
print("  Pipeline: scan_phase2_rooms -> prepare_df_phase2 (RMT/RMH/RDP merge) -> day loop")
print("  -> _analyze_single_room_core (T/H/P check) -> check_reverse_violations")
print("  -> _compute_plot_result -> Excel Report")
print("  Please wait...\n")

out_path, logs, plot_result = analysis_logic.analyze_files_phase2(
    folder_path=folder_path,
    setpoint_path=setpoint_path,
    selected_rooms=["11-1-012", "11-1-013"],
    start_date="2026-05-01",
    end_date="2026-05-02"
)

print("\n" + "=" * 80)
print("InT-06 DEBUG OUTPUT: Full logs from Phase II system process")
print("=" * 80)
print(logs)

print("\n" + "=" * 80)
print("InT-06: Structured Results")
print("=" * 80)

print("--- Output Excel Report ---")
if out_path:
    print(f"Report generated: {out_path}")
    print(f"File exists on disk: {os.path.exists(out_path)}")
    print(f"File size: {os.path.getsize(out_path):,} bytes")
else:
    print("No report generated")

print("\n--- Manual data inspection: prepare_df_phase2 for 11-1-012 ---")
room_dir = os.path.join(folder_path, "2026-05-01")
room_id, df_p2, sensors = analysis_logic.prepare_df_phase2(room_dir, "11-1-012", setpoint_df)
print(f"Room ID: {room_id}")
print(f"Sensors discovered: {sensors}")
print(f"DataFrame shape: {df_p2.shape}")
print(f"Date range: {df_p2['DateTime'].min()} to {df_p2['DateTime'].max()}")
display(df_p2.head(15))

print("\n--- Plot Result Summary ---")
if isinstance(plot_result, dict):
    print(f"Keys in plot_result: {list(plot_result.keys())}")
    if "summary" in plot_result:
        summary_df = pd.DataFrame(plot_result["summary"])
        display(summary_df)
    if "violation_intervals" in plot_result:
        v_df = pd.DataFrame(plot_result["violation_intervals"])
        if not v_df.empty:
            print("Violation intervals:")
            display(v_df)
    if "stats" in plot_result:
        display(plot_result["stats"])

print("=" * 80)
print("InT-06 COMPLETE")
print("=" * 80)


InT-06: Phase II (EMS) Full Statistical Execution [integration mode]
--- STEP 1: Data Configuration ---
SetPointLimit_B11.xlsx loaded: 323 rooms
Selected rooms: ['11-1-012', '11-1-013']
Date range: 2026-05-01 to 2026-05-02
Folder: D:\\ex_work\\AirQualityReview_Project\\data\\csv_b11

--- STEP 2: Verify Phase II room directories exist ---
  2026-05-01/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-01_08-00-03-661_1.Csv...)
  2026-05-02/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-02_08-00-03-217_1.Csv...)
  2026-05-03/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-03_08-00-02-810_1.Csv...)
  2026-05-04/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-03_08-00-02-810_1.Csv...)
  2026-05-05/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-05_08-00-02-828_1.Csv...)
  2026-05-06/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-06_08-00-03-504_1.Csv...)
  2026-05-07/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-07_08-00-02-689_1.Csv...)
  2026-05-08/: 73 CSV files (e.g. 11-1-012_RDP_2026-05-08_08-00-02-586_1.Csv...)
  2026-05-09

,DateTime,Temperature,Humidity,Pressure
0,2026-04-30 08:00:00,22.47,49.04,15.46
1,2026-04-30 08:05:00,22.44,48.53,14.12
2,2026-04-30 08:10:00,22.44,48.53,16.90
3,2026-04-30 08:15:00,22.40,48.79,15.92
4,2026-04-30 08:20:00,22.40,48.79,15.94
5,2026-04-30 08:25:00,22.20,48.94,16.96
6,2026-04-30 08:30:00,22.20,48.94,16.20
7,2026-04-30 08:35:00,22.14,49.12,12.11
8,2026-04-30 08:40:00,22.14,49.12,14.01
9,2026-04-30 08:45:00,22.01,49.46,7.22



--- Plot Result Summary ---
Keys in plot_result: ['summary', 'plot_data', 'violation_intervals', 'stats', 'room_errors']


,room_id,room_name,temp_v,hum_v,press_v
0,11-1-012,MAL 2,0,4,0
1,11-1-013,PAL 2,0,4,0


Violation intervals:


,room_id,room_name,type,start,end,duration,status
0,11-1-012,MAL 2,Humidity,2026-05-01 10:15:00,2026-05-01 14:45:00,270.0,Low
1,11-1-012,MAL 2,Humidity,2026-05-01 17:30:00,2026-05-01 18:35:00,65.0,Low
2,11-1-012,MAL 2,Humidity,2026-05-01 20:40:00,2026-05-01 21:25:00,45.0,Low
3,11-1-012,MAL 2,Humidity,2026-05-01 22:00:00,2026-05-02 00:00:00,120.0,Low
4,11-1-013,PAL 2,Humidity,2026-05-01 10:15:00,2026-05-01 14:45:00,270.0,Low
5,11-1-013,PAL 2,Humidity,2026-05-01 17:30:00,2026-05-01 18:35:00,65.0,Low
6,11-1-013,PAL 2,Humidity,2026-05-01 20:40:00,2026-05-01 21:25:00,45.0,Low
7,11-1-013,PAL 2,Humidity,2026-05-01 22:00:00,2026-05-02 00:00:00,120.0,Low


{'total': 2, 'passed': 0, 'violations': 2, 'errors': 0}

InT-06 COMPLETE
